# Ear Check: training and export

Trains a MobileNetV3-Small classifier on the Eardrum Dataset (otitis media) and exports it to ONNX for the browser app.

Before running: upload the dataset zip from Kaggle to Google Drive and set `ZIP_PATH` below. Use a GPU runtime (Runtime > Change runtime type > T4 GPU). Run the cells in order.

## 1. Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
ZIP_PATH = '/content/drive/MyDrive/YOUR_FILE.zip'   # <- edit: name of the dataset zip in your Drive

!mkdir -p /content/data
!unzip -q "$ZIP_PATH" -d /content/data

In [ ]:
# folders and file counts
import os
from collections import Counter
for root, dirs, files in os.walk('/content/data'):
    if files:
        exts = Counter(os.path.splitext(f)[1].lower() for f in files)
        print(f"{len(files):5d}  {root}  {dict(exts)}")

In [ ]:
# one sample per class, to check these are eardrum photos
from PIL import Image
import matplotlib.pyplot as plt

root = '/content/data/eardrumDs'
classes = sorted(os.listdir(root))
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for ax, c in zip(axes.flat, classes):
    f = sorted(os.listdir(f'{root}/{c}'))[0]
    im = Image.open(f'{root}/{c}/{f}')
    ax.imshow(im)
    ax.set_title(f'{c} {im.size}')
    ax.axis('off')
plt.show()

## 2. Prepare the dataset

Keeps six classes (three folders with fewer than 20 images are dropped), crops the central 350x350 px of each 500x500 image to remove the black corners of the circular endoscope view, resizes to 224x224 and makes a stratified 70/15/15 split. The web app applies the same crop (central 70 %).

In [ ]:
import os, shutil
from PIL import Image
from sklearn.model_selection import train_test_split

SRC = '/content/data/eardrumDs'
DST = '/content/ds'
KEEP = ['Normal', 'Earwax', 'tympanoskleros', 'Aom', 'Chornic', 'OtitExterna']
shutil.rmtree(DST, ignore_errors=True)

paths, labels = [], []
for c in KEEP:
    for f in sorted(os.listdir(f'{SRC}/{c}')):
        paths.append(f'{SRC}/{c}/{f}')
        labels.append(c)

p_tr, p_tmp, y_tr, y_tmp = train_test_split(paths, labels, test_size=0.30, stratify=labels, random_state=42)
p_va, p_te, y_va, y_te = train_test_split(p_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=42)

for split, ps, ys in [('train', p_tr, y_tr), ('val', p_va, y_va), ('test', p_te, y_te)]:
    for p, y in zip(ps, ys):
        d = f'{DST}/{split}/{y}'
        os.makedirs(d, exist_ok=True)
        im = Image.open(p).convert('RGB')
        im = im.crop((75, 75, 425, 425)).resize((224, 224), Image.BILINEAR)
        im.save(f'{d}/{os.path.splitext(os.path.basename(p))[0]}.png')

for split in ['train', 'val', 'test']:
    print(split, {c: len(os.listdir(f'{DST}/{split}/{c}')) for c in KEEP})

## 3. Train

Stage 1: frozen backbone, new head (10 epochs, lr 1e-3). Stage 2: backbone unfrozen with BatchNorm layers kept frozen (15 epochs, lr 1e-4). Square-root inverse-frequency class weights because the classes are very unbalanced.

In [ ]:
import os, numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

DS = '/content/ds'
def load(split, shuffle):
    return keras.utils.image_dataset_from_directory(
        f'{DS}/{split}', image_size=(224, 224), batch_size=32,
        label_mode='categorical', shuffle=shuffle, seed=1)

train_ds, val_ds, test_ds = load('train', True), load('val', False), load('test', False)
class_names = train_ds.class_names
print(class_names)   # must stay: Aom, Chornic, Earwax, Normal, OtitExterna, tympanoskleros (hard-coded in the app)

counts = np.array([len(os.listdir(f'{DS}/train/{c}')) for c in class_names])
w = np.sqrt(counts.sum() / (len(counts) * counts))
class_weight = {i: float(x) for i, x in enumerate(w)}

aug = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.2),
    layers.RandomContrast(0.2),
])

base = keras.applications.MobileNetV3Small(
    input_shape=(224, 224, 3), include_top=False, weights='imagenet', pooling='avg')
base.trainable = False

inp = keras.Input((224, 224, 3))
x = aug(inp)
x = base(x, training=False)
x = layers.Dropout(0.3)(x)
out = layers.Dense(len(class_names), activation='softmax')(x)
model = keras.Model(inp, out)

model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=10, class_weight=class_weight)

# fine-tuning
base.trainable = True
for l in base.layers:
    if isinstance(l, layers.BatchNormalization):
        l.trainable = False
model.compile(optimizer=keras.optimizers.Adam(1e-4),
              loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=15, class_weight=class_weight)

## 4. Evaluate on the test split

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
y_true = np.concatenate([y.numpy().argmax(1) for _, y in test_ds])
y_pred = model.predict(test_ds).argmax(1)
print(classification_report(y_true, y_pred, target_names=class_names))
print(confusion_matrix(y_true, y_pred))

model.save('/content/drive/MyDrive/ear_model.keras')

## 5. Choose the "no signs" threshold

Shows, for several thresholds, how many diseased ears would be called "ok" and how many healthy ears would trigger an alert. Pick the threshold on VAL, use TEST only as a final check. The app uses 0.6.

In [ ]:
def get(ds):
    y = np.concatenate([yy.numpy().argmax(1) for _, yy in ds])
    p = model.predict(ds, verbose=0)
    return y, p

yv, pv = get(val_ds)
yt, pt = get(test_ds)

idx = {n: i for i, n in enumerate(class_names)}
NORMAL = idx['Normal']
SICK = [idx['Aom'], idx['Chornic'], idx['OtitExterna'], idx['tympanoskleros']]

def stats(y, p, t):
    ok = (p.argmax(1) == NORMAL) & (p[:, NORMAL] >= t)
    sick, normal = np.isin(y, SICK), (y == NORMAL)
    return ok[sick].mean(), (~ok[normal]).mean()

print("thr   | VAL: sick->ok / normal->alert | TEST: sick->ok / normal->alert")
for t in [0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    a, b = stats(yv, pv, t)
    c, d = stats(yt, pt, t)
    print(f" {t:.1f}  |   {a:5.0%} / {b:5.0%}            |    {c:5.0%} / {d:5.0%}")

## 6. Export to ONNX

Builds an inference-only model (no augmentation layers). It takes raw 0-255 RGB images of shape (N, 224, 224, 3), normalisation is inside the network.

In [ ]:
base = [l for l in model.layers if 'mobilenet' in l.name.lower()][0]
dense = model.layers[-1]

inp = keras.Input((224, 224, 3))
x = base(inp, training=False)
inf = keras.Model(inp, dense(x))

p_old = model.predict(test_ds.take(1), verbose=0)
p_new = inf.predict(next(iter(test_ds))[0], verbose=0)
print('max difference vs original model:', np.abs(p_old - p_new).max())

inf.export('/content/ear_savedmodel')

In [ ]:
!pip -q install tf2onnx onnxruntime
!python -m tf2onnx.convert --saved-model /content/ear_savedmodel --output /content/ear.onnx --opset 13
!ls -lh /content/ear.onnx

In [ ]:
import onnxruntime as ort, shutil
sess = ort.InferenceSession('/content/ear.onnx')
name = sess.get_inputs()[0].name
xb = next(iter(test_ds))[0].numpy().astype('float32')
out = sess.run(None, {name: xb})[0]
print('ONNX vs Keras, max difference:', np.abs(out - inf.predict(xb, verbose=0)).max())
print('input:', sess.get_inputs()[0].shape, 'output:', sess.get_outputs()[0].shape)

shutil.copy('/content/ear.onnx', '/content/drive/MyDrive/ear.onnx')

## 7. Optional: demo images for the web app

Two test-split images per class, converted from TIFF to PNG at the original 500x500 size (the web app does its own crop). Zipped to Drive.

In [ ]:
import glob
SRC = '/content/data/eardrumDs'
OUT = '/content/demo'
shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(OUT)

for c in ['Normal', 'Earwax', 'tympanoskleros', 'Aom', 'Chornic', 'OtitExterna']:
    for s in sorted(os.listdir(f'/content/ds/test/{c}'))[:2]:
        stem = os.path.splitext(s)[0]
        src = glob.glob(f'{SRC}/{c}/{stem}.*')[0]
        Image.open(src).convert('RGB').save(f'{OUT}/{c}_{stem}.png')

shutil.make_archive('/content/drive/MyDrive/demo_images', 'zip', OUT)
print(sorted(os.listdir(OUT)))